# Financial Transaction Monitoring AI — Random Forest Alert Engine

Notebook ini adalah **core AI/ML pipeline** untuk sistem Transaction Monitoring berbasis Random Forest.

Tujuan:
- mendeteksi transaksi yang memiliki karakteristik yang perlu diperiksa;
- menghasilkan **ALERT / NO ALERT**;
- tidak melakukan blocking, suspend, close account, atau tindakan enforcement otomatis;
- mempertahankan data sumber dalam kondisi read-only;
- mencegah target leakage dan future-information leakage.

> **Catatan penting:** versi ini menggunakan `scenario_label` hanya sebagai sumber ground truth untuk supervised learning:
> `NORMAL = 0`, `SUSPICIOUS = 1`.
> `EDGE_CASE` dan label lain tidak dipaksakan menjadi suspicious. Baris tersebut tidak dipakai sebagai target supervised, tetapi tetap boleh menjadi bagian dari riwayat transaksi ketika membangun fitur historis.

## 0. Prinsip implementasi

Pipeline ini sengaja dibuat transaction-level terlebih dahulu karena interface masih dibuat di Figma.

Kontrak:
- Observation unit: 1 transaksi.
- Prediction unit: risiko 1 transaksi.
- Alert unit: 1 transaksi.
- Model utama: Random Forest.
- Outer split: 80% data awal → 20% data terbaru secara temporal.
- Validation internal: diambil dari bagian training untuk memilih threshold ALERT.
- Test set: tidak digunakan untuk tuning.
- Output model: score + ALERT/NO ALERT.
- Human review tetap menjadi keputusan akhir.

`customer_status` dan `account_status` hanya merupakan informasi status dan **bukan label AML otomatis**.

In [ ]:
# ============================================================
# 1. IMPORTS & CONFIGURATION
# ============================================================
from pathlib import Path
import json
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    precision_recall_curve
)

warnings.filterwarnings("ignore")

SEED = 42
MODEL_VERSION = "RF_TRANSACTION_ALERT_V1"

# Colab: biasanya file yang di-upload berada di /content.
# Lokal/Jupyter: Path.cwd() akan digunakan.
DATA_DIR = Path.cwd()

DATA_FILES = {
    "transaction": "FCIP_Transaction_350000_FK_FIXED.csv",
    "account": "FCIP_E04_Account_Wallet_FINAL_15000.csv",
    "customer": "customer_10000_fcip_synthetic.csv",
    "device": "FCIP_Device_10000_FINAL.csv",
    "merchant": "merchant_2000_fcip_synthetic.csv",
}

ARTIFACT_DIR = DATA_DIR / "aml_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_FOREST_PARAMS = {
    "n_estimators": 200,
    "random_state": SEED,
    "class_weight": "balanced",
    "max_depth": 12,
    "min_samples_split": 5,
    "n_jobs": -1,
}

print("Data directory :", DATA_DIR)
print("Artifact dir   :", ARTIFACT_DIR)
print("Model version   :", MODEL_VERSION)
print("Random seed     :", SEED)

In [ ]:
# ============================================================
# 2. SAFE DATA DISCOVERY & LOADING
# ============================================================
def resolve_data_file(filename: str) -> Path:
    """Resolve an expected CSV without silently using another file."""
    exact = DATA_DIR / filename
    if exact.exists():
        return exact

    matches = list(DATA_DIR.rglob(filename))
    if len(matches) == 1:
        return matches[0]

    csv_files = sorted(DATA_DIR.rglob("*.csv"))
    raise FileNotFoundError(
        f"File yang diperlukan tidak ditemukan: {filename}\n"
        f"Lokasi yang dicari: {DATA_DIR}\n\n"
        f"CSV yang ditemukan ({len(csv_files)}):\n" +
        "\n".join(f" - {p}" for p in csv_files[:50])
    )

def load_csv_read_only(filename: str) -> pd.DataFrame:
    path = resolve_data_file(filename)
    df = pd.read_csv(path)
    print(f"Loaded: {filename:<45} shape={df.shape}")
    return df.copy()

trx_df = load_csv_read_only(DATA_FILES["transaction"])
acc_df = load_csv_read_only(DATA_FILES["account"])
cust_df = load_csv_read_only(DATA_FILES["customer"])
dev_df = load_csv_read_only(DATA_FILES["device"])
merch_df = load_csv_read_only(DATA_FILES["merchant"])

print("\nSemua tabel berhasil dimuat dalam mode read-only.")

In [ ]:
# ============================================================
# 3. DATA AUDIT
# ============================================================
def audit_table(name: str, df: pd.DataFrame, pk_col: str | None = None) -> dict:
    result = {
        "table": name,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_rows": int(df.duplicated().sum()),
    }

    print("\n" + "=" * 72)
    print(f"TABLE: {name}")
    print("=" * 72)
    print(f"Rows    : {len(df):,}")
    print(f"Columns : {len(df.columns):,}")
    print(f"Duplicate full rows: {df.duplicated().sum():,}")

    if pk_col is not None:
        if pk_col not in df.columns:
            raise KeyError(f"PK '{pk_col}' tidak ditemukan pada tabel {name}.")
        null_pk = int(df[pk_col].isna().sum())
        dup_pk = int(df[pk_col].duplicated().sum())
        result["pk_null"] = null_pk
        result["pk_duplicate"] = dup_pk

        print(f"PK '{pk_col}':")
        print(f"  NULL      : {null_pk:,}")
        print(f"  Duplicate : {dup_pk:,}")
        print(f"  Unique    : {df[pk_col].nunique(dropna=True):,}")

        if null_pk > 0 or dup_pk > 0:
            raise ValueError(
                f"Primary key problem pada {name}: "
                f"NULL={null_pk}, duplicate={dup_pk}."
            )

    print("\nMissing values:")
    miss = df.isna().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    if miss.empty:
        print("  Tidak ada missing value.")
    else:
        for col, count in miss.items():
            print(f"  {col}: {count:,} ({count/len(df)*100:.2f}%)")

    print("\nDtypes:")
    print(df.dtypes.to_string())

    return result

audit_results = []
audit_results.append(audit_table("Customer", cust_df, "customer_id"))
audit_results.append(audit_table("Account", acc_df, "account_id"))
audit_results.append(audit_table("Device", dev_df, "device_id"))
audit_results.append(audit_table("Merchant", merch_df, "merchant_id"))
audit_results.append(audit_table("Transaction", trx_df, "transaction_id"))

audit_summary_df = pd.DataFrame(audit_results)
audit_summary_df

In [ ]:
# ============================================================
# 4. FOREIGN KEY & ONE-TO-MANY VALIDATION
# ============================================================
def validate_fk(
    child_df: pd.DataFrame,
    parent_df: pd.DataFrame,
    fk_col: str,
    pk_col: str,
    relation_name: str
) -> dict:
    if fk_col not in child_df.columns:
        print(f"[SKIP] {relation_name}: {fk_col} tidak tersedia.")
        return {"relation": relation_name, "status": "SKIP"}

    if pk_col not in parent_df.columns:
        raise KeyError(f"PK '{pk_col}' tidak ditemukan pada parent table.")

    parent_keys = set(parent_df[pk_col].dropna().astype(str))
    child_nonnull = child_df[fk_col].dropna().astype(str)

    valid = int(child_nonnull.isin(parent_keys).sum())
    total_nonnull = int(len(child_nonnull))
    null_count = int(child_df[fk_col].isna().sum())
    orphan = total_nonnull - valid

    result = {
        "relation": relation_name,
        "child_rows": len(child_df),
        "nonnull_fk": total_nonnull,
        "null_fk": null_count,
        "valid_fk": valid,
        "orphan_fk": orphan,
        "valid_rate": (valid / total_nonnull * 100) if total_nonnull else np.nan
    }

    print(f"\n{relation_name}")
    print(f"  child rows : {len(child_df):,}")
    print(f"  NULL FK    : {null_count:,}")
    print(f"  valid FK   : {valid:,}")
    print(f"  orphan FK  : {orphan:,}")
    print(f"  valid rate : {result['valid_rate']:.2f}%" if total_nonnull else "  valid rate : N/A")

    return result

fk_results = [
    validate_fk(acc_df, cust_df, "customer_id", "customer_id", "Account -> Customer"),
    validate_fk(trx_df, acc_df, "account_id", "account_id", "Transaction -> Account"),
    validate_fk(trx_df, dev_df, "device_id", "device_id", "Transaction -> Device"),
    validate_fk(trx_df, merch_df, "merchant_id", "merchant_id", "Transaction -> Merchant"),
]

fk_audit_df = pd.DataFrame(fk_results)
fk_audit_df

In [ ]:
# One-to-many sanity checks.
# These checks DO NOT remap IDs and DO NOT create new master records.

for fk_col, parent_name in [
    ("account_id", "Account"),
    ("merchant_id", "Merchant"),
    ("device_id", "Device"),
]:
    if fk_col in trx_df.columns:
        counts = trx_df[fk_col].value_counts(dropna=True)
        print(
            f"{parent_name}: "
            f"{counts.size:,} distinct referenced IDs; "
            f"max transactions per ID = {counts.max():,}; "
            f"median = {counts.median():.1f}"
        )

print("\nOne-to-many structure dipertahankan.")

## 5. Modeling Contract & Ground Truth

Pada versi ini, sistem sengaja dibuat sebagai **binary alert detector**:

- `NORMAL` → `0` (tidak menjadi target alert)
- `SUSPICIOUS` → `1` (kandidat alert)
- `EDGE_CASE` / label lain → tidak dipaksa menjadi `0` atau `1`

Alasannya: `EDGE_CASE` tidak boleh otomatis dianggap AML/suspicious hanya karena bukan `NORMAL`.

`scenario_label` boleh menjadi **sumber ground truth**, tetapi **tidak boleh menjadi fitur X**.

`synthetic_pattern` juga tidak boleh menjadi fitur X.

Jika mapping label tidak tersedia, notebook berhenti daripada mengarang ground truth.

In [ ]:
# ============================================================
# 6. TARGET / GROUND-TRUTH VALIDATION
# ============================================================
required_target_cols = {"scenario_label", "synthetic_pattern"}
missing_target_cols = required_target_cols - set(trx_df.columns)
if missing_target_cols:
    raise KeyError(
        f"Kolom ground-truth yang diperlukan tidak tersedia: {sorted(missing_target_cols)}"
    )

trx_df["scenario_label"] = (
    trx_df["scenario_label"]
    .astype("string")
    .str.strip()
    .str.upper()
)

trx_df["synthetic_pattern"] = (
    trx_df["synthetic_pattern"]
    .astype("string")
    .str.strip()
    .str.upper()
)

print("scenario_label distribution:")
print(trx_df["scenario_label"].value_counts(dropna=False))

print("\nsynthetic_pattern distribution:")
print(trx_df["synthetic_pattern"].value_counts(dropna=False).head(30))

KNOWN_SUPERVISED_LABELS = {"NORMAL", "SUSPICIOUS"}

trx_df["alert_target"] = np.nan
trx_df.loc[trx_df["scenario_label"] == "NORMAL", "alert_target"] = 0
trx_df.loc[trx_df["scenario_label"] == "SUSPICIOUS", "alert_target"] = 1

known_rows = int(trx_df["scenario_label"].isin(KNOWN_SUPERVISED_LABELS).sum())
unknown_rows = len(trx_df) - known_rows

print("\nGround-truth contract:")
print("  NORMAL       -> 0")
print("  SUSPICIOUS   -> 1")
print("  Other labels -> excluded from supervised target")
print(f"  Supervised rows: {known_rows:,}")
print(f"  Unlabeled/other: {unknown_rows:,}")

if trx_df["alert_target"].notna().sum() == 0:
    raise ValueError("Tidak ada baris NORMAL/SUSPICIOUS yang dapat digunakan sebagai ground truth.")

print("\nTarget distribution among supervised rows:")
print(
    trx_df.loc[trx_df["alert_target"].notna(), "alert_target"]
    .value_counts()
    .sort_index()
)

In [ ]:
# ============================================================
# 7. TIMESTAMP AUDIT
# ============================================================
TIMESTAMP_COL = "value_datetime_utc"
AMOUNT_COL = "amount_idr"

for required in [TIMESTAMP_COL, AMOUNT_COL, "account_id"]:
    if required not in trx_df.columns:
        raise KeyError(f"Kolom wajib untuk feature engineering tidak ditemukan: {required}")

trx_df[TIMESTAMP_COL] = pd.to_datetime(
    trx_df[TIMESTAMP_COL],
    errors="coerce",
    utc=True
)

trx_df[AMOUNT_COL] = pd.to_numeric(trx_df[AMOUNT_COL], errors="coerce")

invalid_ts = int(trx_df[TIMESTAMP_COL].isna().sum())
invalid_amount = int(trx_df[AMOUNT_COL].isna().sum())

print("Timestamp audit")
print(f"  Invalid timestamp: {invalid_ts:,} / {len(trx_df):,}")
print(f"  Invalid amount   : {invalid_amount:,} / {len(trx_df):,}")

if invalid_ts > 0:
    raise ValueError("Terdapat timestamp invalid. Perbaiki sumber data sebelum modeling.")

if invalid_amount > 0:
    raise ValueError("Terdapat amount non-numeric/invalid. Perbaiki sumber data sebelum modeling.")

trx_df = trx_df.sort_values(
    [TIMESTAMP_COL, "transaction_id"],
    kind="mergesort"
).reset_index(drop=True)

min_time = trx_df[TIMESTAMP_COL].min()
max_time = trx_df[TIMESTAMP_COL].max()

print(f"Range waktu: {min_time} -> {max_time}")
print(f"Timespan   : {max_time - min_time}")

## 8. Feature Engineering — Point-in-Time Safe

Semua fitur historis hanya menggunakan transaksi sebelum transaksi saat ini.

Fitur yang dibuat:

- `amount_idr`
- `account_txn_count_before`
- `account_cum_amount_before`
- `account_avg_amount_before`
- `count_1h`
- `count_24h`
- `amount_1h`
- `amount_24h`
- `merchant_txn_count_before` bila `merchant_id` tersedia
- `device_txn_count_before` bila `device_id` tersedia
- `txn_hour`
- `txn_dayofweek`

Untuk rolling window digunakan `closed='left'`, sehingga transaksi saat ini dan transaksi pada timestamp yang sama tidak dimasukkan ke riwayatnya sendiri.

`scenario_label`, `synthetic_pattern`, customer status, dan account status tidak digunakan sebagai feature X pada model ini.

In [ ]:
# ============================================================
# 9. POINT-IN-TIME SAFE FEATURE ENGINEERING
# ============================================================
def add_prior_count(
    df: pd.DataFrame,
    group_col: str,
    time_col: str,
    output_col: str
) -> pd.DataFrame:
    """Count transactions strictly earlier than the current timestamp."""
    if group_col not in df.columns:
        return df

    rank_values = (
        df.groupby(group_col, dropna=False)[time_col]
        .rank(method="min", ascending=True)
        - 1
    )

    df[output_col] = rank_values.astype(float)
    df.loc[df[group_col].isna(), output_col] = 0.0
    return df

def add_prior_cumulative_amount(
    df: pd.DataFrame,
    group_col: str,
    time_col: str,
    amount_col: str,
    output_col: str
) -> pd.DataFrame:
    """
    Calculate amount accumulated strictly BEFORE the current timestamp.
    All transactions sharing the same timestamp are excluded from each
    other's historical amount.
    """
    grouped = (
        df.groupby([group_col, time_col], dropna=False, as_index=False)[amount_col]
        .sum()
        .rename(columns={amount_col: "__timestamp_amount"})
    )

    grouped = grouped.sort_values(
        [group_col, time_col],
        kind="mergesort"
    )

    grouped["__cum_before"] = (
        grouped.groupby(group_col, dropna=False)["__timestamp_amount"]
        .cumsum()
        - grouped["__timestamp_amount"]
    )

    helper = grouped[
        [group_col, time_col, "__cum_before"]
    ].copy()

    df = df.merge(
        helper,
        on=[group_col, time_col],
        how="left",
        validate="many_to_one"
    )

    df[output_col] = df["__cum_before"].fillna(0.0)
    df.drop(columns=["__cum_before"], inplace=True)

    return df

def add_prior_rolling_features(
    df: pd.DataFrame,
    group_col: str,
    time_col: str,
    amount_col: str,
    windows=("1h", "24h"),
) -> pd.DataFrame:
    if group_col not in df.columns:
        return df

    temp = df[[group_col, time_col, "transaction_id", amount_col]].copy()
    temp = temp.sort_values(
        [group_col, time_col, "transaction_id"],
        kind="mergesort"
    )

    for window in windows:
        count_values = (
            temp.groupby(group_col, dropna=False)
            .rolling(window, on=time_col, closed="left")["transaction_id"]
            .count()
            .reset_index(drop=True)
            .to_numpy()
        )

        amount_values = (
            temp.groupby(group_col, dropna=False)
            .rolling(window, on=time_col, closed="left")[amount_col]
            .sum()
            .reset_index(drop=True)
            .to_numpy()
        )

        temp[f"__count_{window}"] = np.nan_to_num(
            count_values,
            nan=0.0
        )
        temp[f"__amount_{window}"] = np.nan_to_num(
            amount_values,
            nan=0.0
        )

    feature_cols_local = [
        c for c in temp.columns
        if c.startswith("__count_") or c.startswith("__amount_")
    ]

    helper = temp[
        [group_col, time_col, "transaction_id"] + feature_cols_local
    ].copy()

    df = df.merge(
        helper,
        on=[group_col, time_col, "transaction_id"],
        how="left",
        validate="one_to_one"
    )

    for col in feature_cols_local:
        public_name = col.replace("__", "")
        df[public_name] = df[col].fillna(0.0)
        df.drop(columns=[col], inplace=True)

    df = df.sort_values(
        [time_col, "transaction_id"],
        kind="mergesort"
    ).reset_index(drop=True)

    return df

# Strictly prior account activity.
trx_df = add_prior_count(
    trx_df,
    "account_id",
    TIMESTAMP_COL,
    "account_txn_count_before"
)

trx_df = add_prior_cumulative_amount(
    trx_df,
    "account_id",
    TIMESTAMP_COL,
    AMOUNT_COL,
    "account_cum_amount_before"
)

trx_df["account_avg_amount_before"] = np.where(
    trx_df["account_txn_count_before"] > 0,
    trx_df["account_cum_amount_before"]
    / trx_df["account_txn_count_before"],
    0.0
)

# Rolling windows are closed on the left: current/same-timestamp rows
# are excluded from the historical window.
trx_df = add_prior_rolling_features(
    trx_df,
    group_col="account_id",
    time_col=TIMESTAMP_COL,
    amount_col=AMOUNT_COL,
    windows=("1h", "24h"),
)

if "merchant_id" in trx_df.columns:
    trx_df = add_prior_count(
        trx_df,
        "merchant_id",
        TIMESTAMP_COL,
        "merchant_txn_count_before"
    )

if "device_id" in trx_df.columns:
    trx_df = add_prior_count(
        trx_df,
        "device_id",
        TIMESTAMP_COL,
        "device_txn_count_before"
    )

trx_df["txn_hour"] = trx_df[TIMESTAMP_COL].dt.hour.astype(int)
trx_df["txn_dayofweek"] = trx_df[TIMESTAMP_COL].dt.dayofweek.astype(int)

numeric_candidates = [
    "account_txn_count_before",
    "account_cum_amount_before",
    "account_avg_amount_before",
    "count_1h",
    "amount_1h",
    "count_24h",
    "amount_24h",
    "merchant_txn_count_before",
    "device_txn_count_before",
    "txn_hour",
    "txn_dayofweek",
]

for col in numeric_candidates:
    if col in trx_df.columns:
        trx_df[col] = pd.to_numeric(
            trx_df[col],
            errors="coerce"
        ).fillna(0.0)

print("Feature engineering selesai.")

In [ ]:
# ============================================================
# 10. FEATURE CONTRACT & DATA LINEAGE
# ============================================================
candidate_features = [
    AMOUNT_COL,
    "account_txn_count_before",
    "account_cum_amount_before",
    "account_avg_amount_before",
    "count_1h",
    "count_24h",
    "amount_1h",
    "amount_24h",
    "merchant_txn_count_before",
    "device_txn_count_before",
    "txn_hour",
    "txn_dayofweek",
]

feature_cols = [c for c in candidate_features if c in trx_df.columns]

FORBIDDEN_FEATURES = {
    "scenario_label",
    "synthetic_pattern",
    "alert_target",
    "customer_status",
    "account_status",
}

leaked = FORBIDDEN_FEATURES.intersection(feature_cols)
if leaked:
    raise AssertionError(
        f"Leakage: feature terlarang masuk X: {sorted(leaked)}"
    )

if not feature_cols:
    raise ValueError("Tidak ada feature yang valid untuk model.")

source_basis = {
    AMOUNT_COL: "Transaction.amount_idr",
    "account_txn_count_before": "Prior account transaction history",
    "account_cum_amount_before": "Prior account transaction amounts",
    "account_avg_amount_before": "Prior account amount average",
    "count_1h": "Prior 1-hour account activity",
    "count_24h": "Prior 24-hour account activity",
    "amount_1h": "Prior 1-hour account amount",
    "amount_24h": "Prior 24-hour account amount",
    "merchant_txn_count_before": "Prior merchant transaction history",
    "device_txn_count_before": "Prior device transaction history",
    "txn_hour": "Transaction timestamp hour",
    "txn_dayofweek": "Transaction timestamp weekday",
}

feature_lineage = pd.DataFrame({
    "feature": feature_cols,
    "source_or_basis": [source_basis[f] for f in feature_cols]
})

print("Model features:")
print(feature_cols)
print("\nForbidden feature leakage audit: PASS")
feature_lineage

In [ ]:
# ============================================================
# 11. TEMPORAL 80:20 OUTER SPLIT
# ============================================================
def temporal_cutoff(
    df: pd.DataFrame,
    time_col: str,
    train_fraction: float = 0.80
):
    """Choose a unique timestamp boundary near the requested fraction."""
    unique_times = (
        df[time_col]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    if len(unique_times) < 2:
        raise ValueError("Tidak cukup unique timestamp untuk temporal split.")

    cutoff_idx = min(
        max(int(len(unique_times) * train_fraction), 1),
        len(unique_times) - 1
    )

    return unique_times.iloc[cutoff_idx]

outer_cutoff = temporal_cutoff(
    trx_df,
    TIMESTAMP_COL,
    train_fraction=0.80
)

train_all = trx_df[
    trx_df[TIMESTAMP_COL] < outer_cutoff
].copy()

test_all = trx_df[
    trx_df[TIMESTAMP_COL] >= outer_cutoff
].copy()

print("OUTER TEMPORAL SPLIT")
print(f"Requested ratio : 80% / 20%")
print(f"Actual train    : {len(train_all):,} ({len(train_all)/len(trx_df)*100:.2f}%)")
print(f"Actual test     : {len(test_all):,} ({len(test_all)/len(trx_df)*100:.2f}%)")
print(f"Cutoff          : {outer_cutoff}")
print(
    f"Train time      : {train_all[TIMESTAMP_COL].min()} "
    f"-> {train_all[TIMESTAMP_COL].max()}"
)
print(
    f"Test time       : {test_all[TIMESTAMP_COL].min()} "
    f"-> {test_all[TIMESTAMP_COL].max()}"
)

if train_all.empty or test_all.empty:
    raise ValueError("Temporal split menghasilkan train/test kosong.")

if train_all[TIMESTAMP_COL].max() >= test_all[TIMESTAMP_COL].min():
    raise AssertionError("FAILED: temporal overlap.")
else:
    print("Temporal boundary: PASS")

In [ ]:
# ============================================================
# 12. VALIDATION SPLIT INSIDE TRAINING
# ============================================================
# Validation mengambil timestamp paling baru dari 20% bagian training.
# Final test tetap untouched untuk evaluasi terakhir.

validation_cutoff = temporal_cutoff(
    train_all,
    TIMESTAMP_COL,
    train_fraction=0.80
)

train_fit = train_all[
    train_all[TIMESTAMP_COL] < validation_cutoff
].copy()

val_df = train_all[
    train_all[TIMESTAMP_COL] >= validation_cutoff
].copy()

train_fit = train_fit[train_fit["alert_target"].notna()].copy()
val_df = val_df[val_df["alert_target"].notna()].copy()
test_eval = test_all[test_all["alert_target"].notna()].copy()

for name, frame in [
    ("train_fit", train_fit),
    ("validation", val_df),
    ("test", test_eval),
]:
    classes = frame["alert_target"].astype(int).nunique()
    print(f"{name:<12}: {len(frame):,} rows, classes={classes}")

    if classes < 2:
        raise ValueError(
            f"{name} hanya mempunyai satu kelas. "
            "Tidak aman memaksa model/threshold. "
            "Periksa distribusi label atau struktur waktu dataset."
        )

print(f"Validation cutoff: {validation_cutoff}")
print("\nValidation is used only for threshold selection.")
print("Final test set remains untouched.")

In [ ]:
# ============================================================
# 13. PREPARE X/y
# ============================================================
X_train_fit = train_fit[feature_cols].copy()
y_train_fit = train_fit["alert_target"].astype(int)

X_val = val_df[feature_cols].copy()
y_val = val_df["alert_target"].astype(int)

train_full_labeled = train_all[train_all["alert_target"].notna()].copy()

X_train_full = train_full_labeled[feature_cols].copy()
y_train_full = train_full_labeled["alert_target"].astype(int)

X_test = test_eval[feature_cols].copy()
y_test = test_eval["alert_target"].astype(int)

print("Shapes:")
print("  X_train_fit :", X_train_fit.shape)
print("  X_val       :", X_val.shape)
print("  X_train_full:", X_train_full.shape)
print("  X_test      :", X_test.shape)

In [ ]:
# ============================================================
# 14. RANDOM FOREST — VALIDATION MODEL
# ============================================================
rf_for_validation = RandomForestClassifier(**RANDOM_FOREST_PARAMS)

start = time.time()
rf_for_validation.fit(X_train_fit, y_train_fit)
fit_seconds = time.time() - start

val_scores = rf_for_validation.predict_proba(X_val)[:, 1]

print(f"Validation model trained in {fit_seconds:.2f}s")

In [ ]:
# ============================================================
# 15. VALIDATION-BASED ALERT THRESHOLD
# ============================================================
precision, recall, thresholds = precision_recall_curve(
    y_val,
    val_scores
)

# Precision/recall memiliki satu elemen lebih banyak daripada thresholds.
f1_values = (
    2 * precision[:-1] * recall[:-1]
) / np.maximum(
    precision[:-1] + recall[:-1],
    1e-12
)

best_idx = int(np.nanargmax(f1_values))
ALERT_THRESHOLD = float(thresholds[best_idx])

val_pred = (val_scores >= ALERT_THRESHOLD).astype(int)

print(f"Selected ALERT threshold: {ALERT_THRESHOLD:.6f}")
print(
    f"Validation Precision: "
    f"{precision_score(y_val, val_pred, zero_division=0):.4f}"
)
print(
    f"Validation Recall   : "
    f"{recall_score(y_val, val_pred, zero_division=0):.4f}"
)
print(
    f"Validation F1       : "
    f"{f1_score(y_val, val_pred, zero_division=0):.4f}"
)
print(
    f"Validation PR-AUC   : "
    f"{average_precision_score(y_val, val_scores):.4f}"
)

print(
    "\nThreshold dipilih dari validation set; "
    "final test set belum disentuh."
)

In [ ]:
# ============================================================
# 16. FINAL RANDOM FOREST — ALL 80% TRAINING
# ============================================================
rf_model = RandomForestClassifier(**RANDOM_FOREST_PARAMS)

start = time.time()
rf_model.fit(X_train_full, y_train_full)
fit_seconds = time.time() - start

print(f"Final Random Forest trained in {fit_seconds:.2f}s")
print(f"Training rows: {len(X_train_full):,}")
print(f"Features     : {feature_cols}")

In [ ]:
# ============================================================
# 17. FINAL TEST EVALUATION
# ============================================================
test_scores = rf_model.predict_proba(X_test)[:, 1]
test_pred = (test_scores >= ALERT_THRESHOLD).astype(int)

print("=" * 72)
print("FINAL TEST EVALUATION")
print("=" * 72)

print("\nClassification report:")
print(
    classification_report(
        y_test,
        test_pred,
        target_names=["NO_ALERT / NORMAL", "ALERT / SUSPICIOUS"],
        digits=4,
        zero_division=0
    )
)

cm = confusion_matrix(y_test, test_pred)

print("Confusion Matrix:")
print(cm)

test_precision = precision_score(
    y_test, test_pred, zero_division=0
)
test_recall = recall_score(
    y_test, test_pred, zero_division=0
)
test_f1 = f1_score(
    y_test, test_pred, zero_division=0
)
test_pr_auc = average_precision_score(
    y_test, test_scores
)
test_alert_rate = float(test_pred.mean())

evaluation_summary = pd.DataFrame([{
    "model_version": MODEL_VERSION,
    "threshold": ALERT_THRESHOLD,
    "precision": test_precision,
    "recall": test_recall,
    "f1": test_f1,
    "pr_auc": test_pr_auc,
    "alert_rate": test_alert_rate,
    "test_rows": len(y_test),
}])

evaluation_summary

In [ ]:
# ============================================================
# 18. GLOBAL MODEL INTERPRETABILITY
# ============================================================
importance_df = (
    pd.DataFrame({
        "feature": feature_cols,
        "importance": rf_model.feature_importances_
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Global feature importance:")
display(importance_df)

top_features = importance_df.head(5)["feature"].tolist()

print("\nTop contributing features (global model importance):")
for rank, feature in enumerate(top_features, start=1):
    print(f"{rank}. {feature}")

print(
    "\nCatatan: feature importance adalah sinyal kontribusi model, "
    "bukan bukti kausal."
)

In [ ]:
# ============================================================
# 19. ALERT OUTPUT — UI READY
# ============================================================
def build_alert_output(
    raw_df: pd.DataFrame,
    feature_df: pd.DataFrame,
    model,
    threshold: float,
    model_version: str
) -> pd.DataFrame:
    """Return interface-ready alert data without changing source data."""
    X = feature_df[feature_cols].copy()
    scores = model.predict_proba(X)[:, 1]
    statuses = np.where(scores >= threshold, "ALERT", "NO ALERT")

    output = pd.DataFrame({
        "transaction_id": raw_df["transaction_id"].values,
        "account_id": raw_df["account_id"].values,
        "timestamp": raw_df[TIMESTAMP_COL].values,
        "model_score": scores,
        "threshold": threshold,
        "alert_status": statuses,
        "model_version": model_version,
    })

    output["reason"] = np.where(
        output["alert_status"].eq("ALERT"),
        "Model score melewati threshold; perlu validasi manusia.",
        "Model score tidak melewati threshold pemeriksaan."
    )

    return output

# UI-ready results for the test-period transactions.
test_output = build_alert_output(
    raw_df=test_eval,
    feature_df=test_eval,
    model=rf_model,
    threshold=ALERT_THRESHOLD,
    model_version=MODEL_VERSION,
)

print("Contoh output untuk interface:")
display(test_output.head(20))

print("\nAlert count:")
print(test_output["alert_status"].value_counts())

In [ ]:
# ============================================================
# 20. ALERT DETAIL FOR HUMAN REVIEW
# ============================================================
# Snapshot fitur disediakan agar reviewer dapat melihat karakteristik
# transaksi yang masuk antrean. Ini bukan causal explanation.

# test_output dan test_eval berasal dari urutan transaksi yang sama.
# Karena itu kita gabungkan berdasarkan row order, bukan timestamp,
# sehingga perubahan nama kolom output tidak menimbulkan merge ambiguity.

alert_details = pd.concat(
    [
        test_output.reset_index(drop=True),
        test_eval[feature_cols].reset_index(drop=True)
    ],
    axis=1
)

alert_review_queue = (
    alert_details[
        alert_details["alert_status"] == "ALERT"
    ]
    .sort_values("model_score", ascending=False)
    .reset_index(drop=True)
)

print(f"Human-review queue: {len(alert_review_queue):,} ALERT")
display(alert_review_queue.head(20))

In [ ]:
# ============================================================
# 21. LEAKAGE / SAFETY ACCEPTANCE TEST
# ============================================================
assert not FORBIDDEN_FEATURES.intersection(feature_cols), (
    "FAILED: forbidden feature leakage detected."
)

assert "scenario_label" not in feature_cols
assert "synthetic_pattern" not in feature_cols
assert "alert_target" not in feature_cols

assert len(X_train_full) > 0
assert len(X_test) > 0

assert set(y_train_full.unique()).issubset({0, 1})
assert set(y_test.unique()).issubset({0, 1})

assert 0.0 < ALERT_THRESHOLD < 1.0

print("PASS: feature leakage checks")
print("PASS: target domain checks")
print("PASS: threshold sanity check")
print("PASS: model output domain checks")
print("PASS: source dataset remains read-only")

In [ ]:
# ============================================================
# 22. SAVE MODEL ARTIFACTS
# ============================================================
import joblib

model_path = ARTIFACT_DIR / "rf_transaction_alert_model.joblib"
metadata_path = ARTIFACT_DIR / "rf_transaction_alert_metadata.json"
evaluation_path = ARTIFACT_DIR / "rf_transaction_alert_evaluation.csv"

joblib.dump(rf_model, model_path)

metadata = {
    "model_version": MODEL_VERSION,
    "seed": SEED,
    "model_type": "RandomForestClassifier",
    "parameters": RANDOM_FOREST_PARAMS,
    "feature_columns": feature_cols,
    "forbidden_features": sorted(FORBIDDEN_FEATURES),
    "target_mapping": {
        "NORMAL": 0,
        "SUSPICIOUS": 1,
        "OTHER": "excluded from supervised target"
    },
    "outer_split": "chronological 80/20",
    "alert_threshold": ALERT_THRESHOLD,
    "threshold_selection": "validation F1 maximization",
    "output_contract": "ALERT / NO ALERT; human validation required",
    "source_dataset_modified": False,
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

evaluation_summary.to_csv(evaluation_path, index=False)

print("Artifacts saved:")
print(f"  Model      : {model_path}")
print(f"  Metadata   : {metadata_path}")
print(f"  Evaluation : {evaluation_path}")

## 23. Cara interface Figma nanti terhubung

Interface tidak perlu mengetahui isi Random Forest.

Alur:

```text
Transaction data
       ↓
Feature engineering
       ↓
Random Forest
       ↓
Model score
       ↓
Threshold
       ↓
ALERT / NO ALERT
       ↓
Human validation
```

Field yang sudah disiapkan untuk interface:

- `transaction_id`
- `account_id`
- `timestamp`
- `model_score`
- `threshold`
- `alert_status`
- `reason`
- `model_version`

Dengan demikian Figma dapat dibuat terpisah dari core AI tanpa mengubah inti model.

**ALERT bukan bukti fraud. NO ALERT bukan bukti aman.**

In [ ]:
# ============================================================
# 24. FINAL SUMMARY
# ============================================================
print("=" * 72)
print("FINAL PIPELINE STATUS")
print("=" * 72)
print("Model                :", MODEL_VERSION)
print("Observation unit     : Transaction")
print("Prediction unit      : Transaction")
print("Alert unit           : Transaction")
print("Model                : Random Forest")
print("Outer split          : 80% chronological train / 20% chronological test")
print(f"Alert threshold      : {ALERT_THRESHOLD:.6f}")
print(f"Test precision       : {test_precision:.4f}")
print(f"Test recall          : {test_recall:.4f}")
print(f"Test F1              : {test_f1:.4f}")
print(f"Test PR-AUC          : {test_pr_auc:.4f}")
print(f"Test alert rate      : {test_alert_rate:.2%}")
print("Automatic enforcement: NO")
print("Human validation     : REQUIRED")
print("Source modification  : NO")
print("=" * 72)